In [0]:
print("Hello Maddhu")

Hello Maddhu


In [0]:
%pip install pydeequ
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("Plant_ID", "plant_01")
plant_id = dbutils.widgets.get("Plant_ID").lower()
print(f"Loading data of {plant_id}")

Loading data of plant-01


In [0]:
# 1. Define the SAS token (do not include the leading '?')
sas_token = "sp=racwdlmeop&st=2025-08-15T06:58:39Z&se=2025-08-15T15:13:39Z&spr=https&sv=2024-11-04&sr=c&sig=IuAO2HGXPqN3SwqydbdpKzk5oZpoPoQg0Jy7CkaRklw%3D"

# 2. Define storage details
storage_account_name = "madhustorage2"
container_name = "input"
# file_name = f"{plant_id}.txt"
file_name = "plant-01.txt"

# 3. Set Spark configs for accessing ADLS Gen2 with SAS token
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net",
    "SAS"
)
spark.conf.set(
    f"fs.azure.sas.token.provider.type.{storage_account_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider"
)
spark.conf.set(
    f"fs.azure.sas.fixed.token.{storage_account_name}.dfs.core.windows.net",
    sas_token
)

# 4. Build the full path to your file using abfss:// format
csv_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{file_name}"

# 5. Read the CSV file into a DataFrame
print(f"Reading IoT data from: {csv_path}")
df = spark.read.csv(csv_path, header=True, inferSchema=True)

# 6. Show the DataFrame
display(df)


Reading IoT data from: abfss://input@madhustorage2.dfs.core.windows.net/plant-01.txt


PlantID,DeviceID,Timestamp,Temperature_C,Humidity_%,Vibration_mm_s,Pressure_bar,Energy_kWh,Status
Plant_01,Dev_001,2025-08-14T00:00:00Z,72.4,45.3,1.2,1.01,15.3,OK
Plant_01,Dev_002,2025-08-14T00:05:00Z,73.1,44.8,1.3,1.02,15.5,OK
Plant_01,Dev_003,2025-08-14T00:10:00Z,71.9,46.0,1.1,1.0,15.1,OK
Plant_01,Dev_001,2025-08-14T00:15:00Z,72.6,45.1,1.2,1.01,15.4,OK
Plant_01,Dev_002,2025-08-14T00:20:00Z,73.0,44.9,1.3,1.02,15.6,OK
Plant_01,Dev_003,2025-08-14T00:25:00Z,71.8,46.2,1.1,1.0,15.0,OK
Plant_01,Dev_001,2025-08-14T00:30:00Z,72.5,45.0,1.2,1.01,15.3,OK
Plant_01,Dev_002,2025-08-14T00:35:00Z,73.2,44.7,1.3,1.02,15.7,OK
Plant_01,Dev_003,2025-08-14T00:40:00Z,71.7,46.1,1.1,1.0,15.1,OK
Plant_01,Dev_001,2025-08-14T00:45:00Z,72.3,45.4,1.2,1.01,15.4,OK


In [0]:
# df = (
#     spark.read
#     .format("csv")
#     .option("header", "true")
#     .load(d_path)
# )
# Define your column names manually
columns = [
    "PlantID",
    "DeviceID",
    "Timestamp_UTC",
    "Temperature_C",
    "Humidity_pct",
    "Vibration_mm_s",
    "Pressure_bar",
    "Energy_kWh",
    "Status"
]

# Read CSV without using the header from file
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .load(csv_path) \
    .toDF(*columns)

df.printSchema()
df.show()



root
 |-- PlantID: string (nullable = true)
 |-- DeviceID: string (nullable = true)
 |-- Timestamp_UTC: string (nullable = true)
 |-- Temperature_C: string (nullable = true)
 |-- Humidity_pct: string (nullable = true)
 |-- Vibration_mm_s: string (nullable = true)
 |-- Pressure_bar: string (nullable = true)
 |-- Energy_kWh: string (nullable = true)
 |-- Status: string (nullable = true)

+--------+--------+--------------------+-------------+------------+--------------+------------+----------+------+
| PlantID|DeviceID|       Timestamp_UTC|Temperature_C|Humidity_pct|Vibration_mm_s|Pressure_bar|Energy_kWh|Status|
+--------+--------+--------------------+-------------+------------+--------------+------------+----------+------+
| PlantID|DeviceID|           Timestamp|Temperature_C|  Humidity_%|Vibration_mm_s|Pressure_bar|Energy_kWh|Status|
|Plant_01| Dev_001|2025-08-14T00:00:00Z|         72.4|        45.3|           1.2|        1.01|      15.3|    OK|
|Plant_01| Dev_002|2025-08-14T00:05:00Z| 

In [0]:
df = df.withColumnRenamed("Timestamp (UTC)", "Timestamp_UTC")

In [0]:
# from pyspark.sql.functions import col, to_timestamp

# df = df.withColumn("Temperature_C", col("Temperature_C").cast("double")) \
#            .withColumn("Humidity_%", col("Humidity_%").cast("double")) \
#            .withColumn("Vibration_mm_s", col("Vibration_mm_s").cast("double")) \
#            .withColumn("Pressure_bar", col("Pressure_bar").cast("double")) \
#            .withColumn("Energy_kWh", col("Energy_kWh").cast("double")) \
#            .withColumn("Timestamp", to_timestamp("Timestamp"))

In [0]:
import os
os.environ["SPARK_VERSION"] = "3.3"
os.environ['PYSPARK_SUBMIT_ARGS'] = '--conf spark.sql.shuffle.partitions=8 pyspark-shell'
import json
from pyspark.sql import SparkSession
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col




In [0]:
check = (
    Check(spark, CheckLevel.Error, "IoT Data Quality Checks")
    .hasSize(lambda x: x >= 3)
    .isPositive("Temperature_C")
    .isNonNegative("Temperature_C")
    .isNonNegative("Humidity_pct")
    .isNonNegative("Vibration_mm_s")
    .isNonNegative("Pressure_bar")
    .isNonNegative("Energy_kWh")
)

In [0]:
result = VerificationSuite(spark).onData(df).addCheck(check).run()
from pydeequ.verification import VerificationResult
result_df = VerificationResult.checkResultsAsDataFrame(spark, result)
VerificationResult.checkResultsAsDataFrame(spark, result).show(truncate=False)
# result_df.show(truncate=False)
display(result_df)

+-----------------------+-----------+------------+--------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------+
|check                  |check_level|check_status|constraint                                                                                                                      |constraint_status|constraint_message|
+-----------------------+-----------+------------+--------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------+
|IoT Data Quality Checks|Error      |Success     |SizeConstraint(Size(None))                                                                                                      |Success          |                  |
|IoT Data Quality Checks|Error      |Success     |ComplianceConstraint(Compliance(Temperature_C is positive,COALESCE(CAST(Temperatur

/databricks/spark/python/pyspark/sql/dataframe.py:163: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


check,check_level,check_status,constraint,constraint_status,constraint_message
IoT Data Quality Checks,Error,Success,SizeConstraint(Size(None)),Success,
IoT Data Quality Checks,Error,Success,"ComplianceConstraint(Compliance(Temperature_C is positive,COALESCE(CAST(Temperature_C AS DECIMAL(20,10)), 1.0) > 0,None))",Success,
IoT Data Quality Checks,Error,Success,"ComplianceConstraint(Compliance(Temperature_C is non-negative,COALESCE(CAST(Temperature_C AS DECIMAL(20,10)), 0.0) >= 0,None))",Success,
IoT Data Quality Checks,Error,Success,"ComplianceConstraint(Compliance(Humidity_pct is non-negative,COALESCE(CAST(Humidity_pct AS DECIMAL(20,10)), 0.0) >= 0,None))",Success,
IoT Data Quality Checks,Error,Success,"ComplianceConstraint(Compliance(Vibration_mm_s is non-negative,COALESCE(CAST(Vibration_mm_s AS DECIMAL(20,10)), 0.0) >= 0,None))",Success,
IoT Data Quality Checks,Error,Success,"ComplianceConstraint(Compliance(Pressure_bar is non-negative,COALESCE(CAST(Pressure_bar AS DECIMAL(20,10)), 0.0) >= 0,None))",Success,
IoT Data Quality Checks,Error,Success,"ComplianceConstraint(Compliance(Energy_kWh is non-negative,COALESCE(CAST(Energy_kWh AS DECIMAL(20,10)), 0.0) >= 0,None))",Success,


In [0]:
result = (
    VerificationSuite(spark)
    .onData(df)
    .addCheck(check)
    .run()
)

In [0]:
result_dict = result.checkResults
report_json = json.dumps(result_dict, indent=  4)


In [0]:
print(report_json)

[
    {
        "check_status": "Success",
        "check_level": "Error",
        "constraint_status": "Success",
        "check": "IoT Data Quality Checks",
        "constraint_message": "",
        "constraint": "SizeConstraint(Size(None))"
    },
    {
        "check_status": "Success",
        "check_level": "Error",
        "constraint_status": "Success",
        "check": "IoT Data Quality Checks",
        "constraint_message": "",
        "constraint": "ComplianceConstraint(Compliance(Temperature_C is positive,COALESCE(CAST(Temperature_C AS DECIMAL(20,10)), 1.0) > 0,None))"
    },
    {
        "check_status": "Success",
        "check_level": "Error",
        "constraint_status": "Success",
        "check": "IoT Data Quality Checks",
        "constraint_message": "",
        "constraint": "ComplianceConstraint(Compliance(Temperature_C is non-negative,COALESCE(CAST(Temperature_C AS DECIMAL(20,10)), 0.0) >= 0,None))"
    },
    {
        "check_status": "Success",
        "check_